# Chapter 9 Running Interactive Scripts

## 1. Installing DukPy

In [2]:
import dukpy

dukpy.evaljs("2 + 2")

4

In [ ]:
# 책 내용과 달리 dukpy 내부 엔진이 quickjs로 바뀌면서 최신 문법 지원되는 것으로 보임
try:
    print(dukpy.evaljs("let a = 4.2; a * 2"))
except e:
    print("ES6 error", e)

8.4


## 2. Running JavaScript Code

In [9]:
import os
from browser import URL, Tab

with open("test.html", "w") as f:
    f.write('<script src="test.js"></script>')

with open("test.js", "w") as f:
    f.write("var x = 2;\nx + x;")

tab = Tab(600)
tab.load(URL("file://" + os.path.abspath("test.html")))

os.remove("test.html")
os.remove("test.js")

Script returned:  4


### 💡 `<script>` 실행 방식 비교 (Blocking vs defer vs async vs nomodule)

> 실제 브라우저는 `<script>` 태그를 만나는 즉시 HTML 파싱을 중단(Blocking)하고 스크립트를 다운로드/실행하지만, 우리 브라우저는 HTML 트리를 다 만든 뒤 실행하므로 `defer`와 유사하게 동작합니다.

| 방식 | 다운로드 | 실행 타이밍 | 파서 차단 | 실행 순서 보장 | 주 용도 |
| :--- | :--- | :--- | :---: | :---: | :--- |
| **기본 (Blocking)** | 즉시 (동기) | 다운로드 직후 | **O** | **O** | 기본 동작 (구형 방식) |
| **`defer`** | 백그라운드 병렬 | **HTML 파싱 완료 직후** (`DOMContentLoaded` 전) | X | **O** | DOM 제어 및 메인 앱 로직 (권장) |
| **`async`** | 백그라운드 병렬 | **다운로드 완료 즉시** | 실행 시 O | **X** (다운로드 순) | 독립 트래커, 분석 도구 (GA 등) |
| **`nomodule`** | 구형 브라우저만 | 구형 브라우저에서만 | - | **O** | 구형 브라우저용 폴백 (`type="module"`과 쌍으로 사용) |

---

#### 🔍 Speculative Parsing (추측 파싱 / Preload Scanner)
- **개념**: 기본 Blocking 스크립트 실행으로 메인 파서가 멈춰 있을 때, 백그라운드 보조 파서가 뒤쪽 HTML을 미리 훑어 외부 리소스(CSS, 이미지, 후속 JS)를 사전 다운로드하는 최적화 기법.
- **원리**: 선행 스크립트가 `document.write` 등으로 HTML 구조를 뒤엎지 않을 것이라 "추측(Speculate)"하여 네트워크 병목(Waterfall)을 방지. (추측 실패 시 임시 상태 롤백/폐기)
- **브라우저 구현**: 멀티스레드 동기화와 롤백 복잡성 때문에, 본 실습 브라우저는 전체 파싱 후 일괄 실행(`defer` 의미론)으로 단순화함.

---

#### 🌐 현대 웹(SPA vs SSR)과의 연관성 및 프레임워크별 해결 전략
- **SPA (CSR)**: 번들러가 `type="module"`/`defer`를 자동 적용해 파서 블로킹은 없으나, HTML이 껍데기(`<div id="root"></div>`)뿐이라 Speculative Parser가 리소스를 미리 찾지 못하고 거대 번들 로드 전까지 빈 화면(흰 화면 병목) 발생.
- **SSR**: 완성된 HTML을 주므로 Speculative Parser가 리소스를 초고속 사전 로딩함. 단, 화면은 보이지만 JS가 로드되어 이벤트 리스너를 결합하기 전까지 동작하지 않는 **하이드레이션(Hydration) 병목**이 현대의 주요 과제로 등장.
- **Vue / Nuxt**: 컴파일러 정적 호이스팅(불변 VNode 하이드레이션 제외), 비동기 컴포넌트(`defineAsyncComponent`), 뷰포트 기반 지연 하이드레이션(Lazy Hydration).
- **Angular**: 비파괴적 하이드레이션(DOM 재사용), `@defer` 블록(뷰포트/인터랙션 조건부 지연 로딩), 이벤트 리플레이(하이드레이션 전 클릭 이벤트 캡처 후 재생).
- **React / Next.js**: 스트리밍 SSR + 선택적 하이드레이션(Suspense 경계별 점진 하이드레이션), 서버 컴포넌트(RSC, 클라이언트에 JS 0바이트 전송).
- **신흥 아키텍처 (Astro / Qwik)**: 아일랜드 아키텍처(필요한 컴포넌트만 독립 하이드레이션), Resumability(서버 상태 직렬화로 하이드레이션 자체를 생략).

## 3. Exporting Functions

In [ ]:
import os
from browser import URL, Tab

# 1. a.js에서 변수 선언, b.js에서 해당 변수를 읽고 console.log()로 출력
with open("a.js", "w") as f:
    f.write("var x = 20;")

with open("b.js", "w") as f:
    f.write('console.log("Hi from JS! x + x = " + (x + x));')

with open("page.html", "w") as f:
    f.write('<script src="a.js"></script><script src="b.js"></script>')

# 2. 브라우저 탭에서 로드하여 상태 공유 및 console.log 동작 검증
tab = Tab(600)
tab.load(URL("file://" + os.path.abspath("page.html")))

# 3. 임시 파일 정리
os.remove("a.js")
os.remove("b.js")
os.remove("page.html")

### 💡 [참고] 호출 구조, Host Bindings vs Polyfill

#### 1. 파이썬-자바스크립트 간 호출 구조
브라우저가 파이썬 함수(`print`)를 `export_function("log", print)`로 내보내면, JS 엔진의 C 브리지를 거쳐 웹 표준 `console.log`로 래핑됨.

```text
[1] 웹 스크립트         : console.log("Hello") 호출
         ↓
[2] 브라우저 JS 런타임   : runtime.js (call_python("log", "Hello") 래핑)
         ↓ (JSON 직렬화 및 C-FFI 바운더리)
[3] C 언어 JS 엔진      : DukPy C 브리지 (수출된 함수 맵에서 "log" 탐색)
         ↓
[4] 브라우저 호스트      : Python의 print("Hello") 실행
```

#### 2. Host Bindings(내장 바인딩) vs Polyfill(폴리필)
- **호스트 바인딩 (Builtins)**: 브라우저/런타임 개발자가 C++/Python 등 호스트 언어의 저수준 기능을 웹 표준 API 형태로 포장해 엔진에 주입하는 공식 브리지 코드 (우리의 `runtime.js`).
- **폴리필 (Polyfill)**: 웹 서비스 개발자가 구형 브라우저에서 누락된 표준 스펙을 메우기 위해 웹페이지 레벨에서 끼워 넣는 하위 호환성 패치 코드 (`core-js` 등).

#### 3. 실제 브라우저 및 런타임 소스코드 사례
- **WebKit (Safari)**: 엔진 내부 [`Source/JavaScriptCore/builtins/`](https://github.com/WebKit/WebKit/tree/main/Source/JavaScriptCore/builtins) 디렉토리에 [`ArrayPrototype.js`](https://github.com/WebKit/WebKit/blob/main/Source/JavaScriptCore/builtins/ArrayPrototype.js) 등 수많은 내장 JS가 들어있고, C++ 엔진 기능을 `@` 접두사 내부 함수로 호출함.
- **Firefox (SpiderMonkey)**: 엔진 내부 [](https://searchfox.org/mozilla-central/source/js/src/builtin/) 디렉토리에 [](https://searchfox.org/mozilla-central/source/js/src/builtin/Array.js) 등 표준 스펙을 JS로 구현한 **Self-hosted JavaScript**가 위치하며, C++ 엔진 기능은 [](https://searchfox.org/mozilla-central/source/js/src/vm/SelfHosting.cpp)의 특수 Intrinsic 함수로 호출함.
- **Node.js**: [`lib/internal/console/constructor.js`](https://github.com/nodejs/node/blob/main/lib/internal/console/constructor.js)에서 JS로 `Console`을 구현하고, 저수준 C++ 시스템 콜(`internalBinding('fs').writeBuffer`)을 호출함.
- **Chromium (V8)**: 과거에는 JS Builtin을 썼으나, 프로토타입 오염 방어와 JIT 성능을 위해 자체 정적 타입 DSL인 [**Torque**](https://v8.dev/docs/torque)를 개발하여 [`src/builtins/*.tq`](https://github.com/v8/v8/tree/main/src/builtins) (예: [`array-slice.tq`](https://github.com/v8/v8/blob/main/src/builtins/array-slice.tq)) 파일들을 초고속 기계어로 직접 컴파일함.